# 05 Simulation Study

This notebook evaluates reconstruction and pooled survival estimation under repeated simulated multi-trial settings.

We study:
1. reconstruction accuracy
2. trial heterogeneity
3. pooled survival contrasts

In [1]:
import pandas as pd
import numpy as np
from src.simulation.trial_generator import simulate_multi_trial
from src.simulation.km_renderer import build_km_render_from_ipd
from src.cen_km.reconstruct import reconstruct_ipd_with_censoring
from src.external_bridge.resolve_ipd_adapter import reconstruct_ipd_with_resolve
from src.validation.metrics import summary_metrics
from src.causal.estimands import trial_level_survival_difference, pooled_survival_difference

In [2]:
# define a replication function (we may port a lot of 05 to /experiment_runner.py)
def run_single_replication(seed: int, t0: float = 12.0):
    multi = simulate_multi_trial(n_trials=4, n_per_trial=250, seed=seed)
    df_multi = multi.ipd.copy()

    # pick one trial for reconstruction benchmarking
    trial_id = df_multi["trial_id"].unique()[0]
    df_trial = df_multi.loc[df_multi["trial_id"] == trial_id].copy()

    render = build_km_render_from_ipd(df_trial, time_col="time", event_col="event")

    ipd_ours = reconstruct_ipd_with_censoring(
        curve_points=render.curve_points,
        censor_points=render.censor_points,
        n_initial=len(df_trial),
    ).ipd

    ipd_resolve = reconstruct_ipd_with_resolve(
        n=len(df_trial),
        t=render.curve_points["time"].tolist(),
        S=render.curve_points["survival"].tolist(),
        cens_t=render.censor_points["time"].tolist(),
        random_state=seed,
        debug=False,
    )

    metrics_ours = summary_metrics(df_trial, ipd_ours, t0=t0)
    metrics_resolve = summary_metrics(df_trial, ipd_resolve, t0=t0)

    trial_effects = trial_level_survival_difference(df_multi, t0=t0)
    pooled = pooled_survival_difference(df_multi, t0=t0)

    return {
        "seed": seed,
        "ours_km_rmse": metrics_ours["km_rmse"],
        "resolve_km_rmse": metrics_resolve["km_rmse"],
        "ours_surv_diff_t0": metrics_ours["survival_diff_t0"],
        "resolve_surv_diff_t0": metrics_resolve["survival_diff_t0"],
        "pooled_delta": pooled["Delta"].iloc[0],
        "trial_delta_sd": trial_effects["Delta"].std(),
    }

In [3]:
results = pd.DataFrame([run_single_replication(seed) for seed in range(100, 120)])
results

,seed,ours_km_rmse,resolve_km_rmse,ours_surv_diff_t0,resolve_surv_diff_t0,pooled_delta,trial_delta_sd
0,100,0.0,0.0,0.0,0.0,0.152184,0.027561
1,101,0.0,0.0,0.0,0.0,0.099239,0.078616
2,102,0.0,0.0,0.0,0.0,0.065937,0.073294
3,103,0.0,0.0,0.0,0.0,0.100741,0.074596
4,104,0.0,0.0,0.0,0.0,0.104409,0.084522
5,105,0.0,0.0,0.0,0.0,0.150517,0.112721
6,106,0.0,0.0,0.0,0.0,0.112769,0.046267
7,107,0.0,0.0,0.0,0.0,0.116031,0.023359
8,108,0.0,0.0,0.0,0.0,0.140564,0.090717
9,109,0.0,0.0,0.0,0.0,0.192519,0.074938


In [4]:
results.describe()

,seed,ours_km_rmse,resolve_km_rmse,ours_surv_diff_t0,resolve_surv_diff_t0,pooled_delta,trial_delta_sd
count,20.00000,20.0,20.0,20.0,20.0,20.000000,20.000000
mean,109.50000,0.0,0.0,0.0,0.0,0.124032,0.064058
std,5.91608,0.0,0.0,0.0,0.0,0.032857,0.026808
min,100.00000,0.0,0.0,0.0,0.0,0.065937,0.012608
25%,104.75000,0.0,0.0,0.0,0.0,0.106476,0.044436
50%,109.50000,0.0,0.0,0.0,0.0,0.114400,0.073929
75%,114.25000,0.0,0.0,0.0,0.0,0.150934,0.080520
max,119.00000,0.0,0.0,0.0,0.0,0.192519,0.112721


## Simulation Study Results

Across 20 simulated multi-trial datasets, both reconstruction methods achieve perfect recovery of the underlying survival process:
- KM RMSE = 0 for all replications  
- zero error in fixed-time survival differences  

This confirms that, under ideal conditions, reconstruction does not introduce additional variability or bias.

---

## Heterogeneity Across Trials

Despite perfect reconstruction, there is substantial variability in treatment effects across trials:
- mean standard deviation of trial-level effects ≈ 0.064  
- ranging from ≈ 0.013 to ≈ 0.113  

This indicates persistent **between-trial heterogeneity**, driven by differences in underlying populations.

---

## Pooled Survival Effects

The pooled survival difference also varies across replications:
- mean pooled effect ≈ 0.124  
- range ≈ 0.066 to 0.193  

This variation reflects changes in trial composition rather than estimation error.

---

## Key Takeaway

These results highlight a central point:

> Once IPD is accurately reconstructed, the dominant source of uncertainty is no longer reconstruction, but heterogeneity across trials.

As a result, naive pooled estimates reflect a mixture of distinct populations and do not correspond to a well-defined causal estimand. This motivates the need for methods that adjust for cross-trial differences and target a common population.